# Klasifikasi Dengan SVM , Naifbayes


In [1]:
!pip install plotly

In [2]:
!pip install -q nltk contractions emoji pyspellchecker Sastrawi

In [3]:
#!pip install gensim


In [4]:
import re
import sys
import time
import string
import warnings
from collections import Counter
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from spellchecker import SpellChecker
import contractions
import emoji
from gensim.models import Word2Vec, FastText
from gensim import corpora, models
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from tqdm import tqdm
warnings.filterwarnings('ignore')

In [5]:
nltk.download("punkt")
nltk.download("punkt_tab")

# Download stopwords
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Load data set

In [6]:
berita = pd.read_csv("Berita.csv")
berita

,No,judul,berita,tanggal,kategori,link
0,1,Airlangga Harap Kenaikan UMP Tingkatkan Daya B...,Menteri Koordinator (Menko) Bidang Perekonomia...,"Minggu, 01 Des 2024 23:40 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412012...
1,2,PT SIER Beri Penghargaan untuk 50 Tenant Terba...,"Dalam rangka memeriahkan hari jadi ke-50, PT S...","Minggu, 01 Des 2024 20:45 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412012...
2,3,Prabowo Bakal Bentuk Kementerian Penerimaan Ne...,Wacana Presiden Prabowo Subianto akan membentu...,"Minggu, 01 Des 2024 19:40 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412011...
3,4,Sinergi Kemenag & BPJS Ketenagakerjaan Lindung...,BPJS Ketenagakerjaan dan Kementerian Agama (Ke...,"Minggu, 01 Des 2024 19:03 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412011...
4,5,Pemerintah Segera Bentuk Satgas PHK Usai Tetap...,Pemerintah akan segera membentuk Satuan Tugas ...,"Minggu, 01 Des 2024 19:00 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412011...
...,...,...,...,...,...,...
1495,1496,Laporan Sebab Tabrakan Pesawat-Black Hawk Dita...,Anggota Dewan Keselamatan Transportasi Nasiona...,"Jumat, 31 Jan 2025 04:40 WIB",Internasional,https://www.cnnindonesia.com/internasional/202...
1496,1497,"Israel Bebaskan 110 Sandera Palestina, Diantar...",Israel telah membebaskan 110 tahanan Palestina...,"Jumat, 31 Jan 2025 03:01 WIB",Internasional,https://www.cnnindonesia.com/internasional/202...
1497,1498,Hamas Konfirmasi Kematian Komandan Al Qassam M...,Hamas mengonfirmasi kematian kepala militernya...,"Jumat, 31 Jan 2025 02:30 WIB",Internasional,https://www.cnnindonesia.com/internasional/202...
1498,1499,Black Box American Airlines Ditemukan Usai Tab...,Tim penyelam diduga menemukan satu dari dua bl...,"Jumat, 31 Jan 2025 01:00 WIB",Internasional,https://www.cnnindonesia.com/internasional/202...


## Fungsi cleaning dengan emoji


In [7]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()  # huruf kecil
    text = emoji.demojize(text)  # ubah emoji jadi teks, misal 😀 -> :grinning_face:
    text = re.sub(r'\d+', '', text)  # hapus angka
    text = text.translate(str.maketrans('', '', string.punctuation))  # hapus tanda baca
    text = re.sub(r'\W', ' ', text)  # hapus karakter non-kata
    text = BeautifulSoup(text, "html.parser").get_text()  # hapus tag HTML
    text = re.sub(r'\s+', ' ', text).strip()  # normalisasi spasi
    return text

# Baca CSV
tempo_df = pd.read_csv("Berita.csv", dtype=str).fillna("")

# Terapkan langsung ke kolom target
tempo_df["isi_berita_clean"] = tempo_df["berita"].apply(clean_text)

# Cleaning kolom target
tempo_df["isi_berita_clean"] = tempo_df["berita"].apply(clean_text)

In [8]:
print("Berita :")
tempo_df[["berita", "isi_berita_clean"]].head(10)

Berita :


,berita,isi_berita_clean
0,Menteri Koordinator (Menko) Bidang Perekonomia...,menteri koordinator menko bidang perekonomian ...
1,"Dalam rangka memeriahkan hari jadi ke-50, PT S...",dalam rangka memeriahkan hari jadi ke pt surab...
2,Wacana Presiden Prabowo Subianto akan membentu...,wacana presiden prabowo subianto akan membentu...
3,BPJS Ketenagakerjaan dan Kementerian Agama (Ke...,bpjs ketenagakerjaan dan kementerian agama kem...
4,Pemerintah akan segera membentuk Satuan Tugas ...,pemerintah akan segera membentuk satuan tugas ...
5,Menko Bidang Infrastruktur dan Pembangunan Kew...,menko bidang infrastruktur dan pembangunan kew...
6,Kepala Badan Gizi Nasional Dadan Hindayana men...,kepala badan gizi nasional dadan hindayana men...
7,Menteri Koordinator Bidang Pangan Zulkifli Has...,menteri koordinator bidang pangan zulkifli has...
8,Uji coba alias commissioning pembangkit listri...,uji coba alias commissioning pembangkit listri...
9,Anak crazy rich pengusaha sawit Kalimantan Sam...,anak crazy rich pengusaha sawit kalimantan sam...


In [9]:
# Tokenisasi untuk PTA
tempo_df["isi_berita_tokens"] = tempo_df["isi_berita_clean"].apply(word_tokenize)

In [10]:
print("\nBerita (isi_berita_tokens):")
tempo_df[["isi_berita_clean", "isi_berita_tokens"]].head(10)


Berita (isi_berita_tokens):


,isi_berita_clean,isi_berita_tokens
0,menteri koordinator menko bidang perekonomian ...,"[menteri, koordinator, menko, bidang, perekono..."
1,dalam rangka memeriahkan hari jadi ke pt surab...,"[dalam, rangka, memeriahkan, hari, jadi, ke, p..."
2,wacana presiden prabowo subianto akan membentu...,"[wacana, presiden, prabowo, subianto, akan, me..."
3,bpjs ketenagakerjaan dan kementerian agama kem...,"[bpjs, ketenagakerjaan, dan, kementerian, agam..."
4,pemerintah akan segera membentuk satuan tugas ...,"[pemerintah, akan, segera, membentuk, satuan, ..."
5,menko bidang infrastruktur dan pembangunan kew...,"[menko, bidang, infrastruktur, dan, pembanguna..."
6,kepala badan gizi nasional dadan hindayana men...,"[kepala, badan, gizi, nasional, dadan, hindaya..."
7,menteri koordinator bidang pangan zulkifli has...,"[menteri, koordinator, bidang, pangan, zulkifl..."
8,uji coba alias commissioning pembangkit listri...,"[uji, coba, alias, commissioning, pembangkit, ..."
9,anak crazy rich pengusaha sawit kalimantan sam...,"[anak, crazy, rich, pengusaha, sawit, kalimant..."


## Stopwords

In [11]:
# Stopwords untuk bahasa Indonesia
stop_words_id = set(stopwords.words('indonesian'))

# Filter stopwords di PTA
tempo_df["isi_berita_filtered"] = tempo_df["isi_berita_tokens"].apply(
    lambda tokens: [word for word in tokens if word not in stop_words_id]
)

In [12]:
print("\nBerita (isi_berita_filtered):")
tempo_df[["isi_berita_tokens", "isi_berita_filtered"]].head(10)


Berita (isi_berita_filtered):


,isi_berita_tokens,isi_berita_filtered
0,"[menteri, koordinator, menko, bidang, perekono...","[menteri, koordinator, menko, bidang, perekono..."
1,"[dalam, rangka, memeriahkan, hari, jadi, ke, p...","[rangka, memeriahkan, pt, surabaya, industrial..."
2,"[wacana, presiden, prabowo, subianto, akan, me...","[wacana, presiden, prabowo, subianto, membentu..."
3,"[bpjs, ketenagakerjaan, dan, kementerian, agam...","[bpjs, ketenagakerjaan, kementerian, agama, ke..."
4,"[pemerintah, akan, segera, membentuk, satuan, ...","[pemerintah, membentuk, satuan, tugas, pemutus..."
5,"[menko, bidang, infrastruktur, dan, pembanguna...","[menko, bidang, infrastruktur, pembangunan, ke..."
6,"[kepala, badan, gizi, nasional, dadan, hindaya...","[kepala, badan, gizi, nasional, dadan, hindaya..."
7,"[menteri, koordinator, bidang, pangan, zulkifl...","[menteri, koordinator, bidang, pangan, zulkifl..."
8,"[uji, coba, alias, commissioning, pembangkit, ...","[uji, coba, alias, commissioning, pembangkit, ..."
9,"[anak, crazy, rich, pengusaha, sawit, kalimant...","[anak, crazy, rich, pengusaha, sawit, kalimant..."


In [13]:
factory = StemmerFactory()
indo_stemmer = factory.create_stemmer()

tempo_df["isi_berita_stemmed"] = tempo_df["isi_berita_filtered"].apply(
    lambda tokens: [indo_stemmer.stem(word) for word in tokens]
)

In [14]:
print("\nBerita - Stemming & Lemmatization (abstrak_id):")
tempo_df[["isi_berita_filtered", "isi_berita_stemmed"]].head(10)


Berita - Stemming & Lemmatization (abstrak_id):


,isi_berita_filtered,isi_berita_stemmed
0,"[menteri, koordinator, menko, bidang, perekono...","[menteri, koordinator, menko, bidang, ekonomi,..."
1,"[rangka, memeriahkan, pt, surabaya, industrial...","[rangka, riah, pt, surabaya, industrial, estat..."
2,"[wacana, presiden, prabowo, subianto, membentu...","[wacana, presiden, prabowo, subianto, bentuk, ..."
3,"[bpjs, ketenagakerjaan, kementerian, agama, ke...","[bpjs, ketenagakerjaan, menteri, agama, kemena..."
4,"[pemerintah, membentuk, satuan, tugas, pemutus...","[perintah, bentuk, satu, tugas, putus, hubung,..."
5,"[menko, bidang, infrastruktur, pembangunan, ke...","[menko, bidang, infrastruktur, bangun, wilayah..."
6,"[kepala, badan, gizi, nasional, dadan, hindaya...","[kepala, badan, gizi, nasional, dad, hindayana..."
7,"[menteri, koordinator, bidang, pangan, zulkifl...","[menteri, koordinator, bidang, pangan, zulkifl..."
8,"[uji, coba, alias, commissioning, pembangkit, ...","[uji, coba, alias, commissioning, bangkit, lis..."
9,"[anak, crazy, rich, pengusaha, sawit, kalimant...","[anak, crazy, rich, usaha, sawit, kalimantan, ..."


## Fungsi expand kontraksi bahasa Indonesia


In [15]:
def expand_indonesian_contractions(text):
  contractions_dict = {
      "gak": "tidak", "ga": "tidak", "nggak": "tidak", "enggak": "tidak", "ngga": "tidak", "gk": "tidak", "tdk": "tidak", "tk": "tidak",
      "gue": "saya", "gw": "saya", "gua": "saya", "sy": "saya", "aq": "saya", "q": "saya", "ane": "saya",
      "lu": "kamu", "loe": "kamu", "lo": "kamu", "km": "kamu", "kmu": "kamu", "elu": "kamu",
      "dah": "sudah", "udah": "sudah", "sdh": "sudah", "udh": "sudah",
      "blm": "belum", "td": "tadi", "ntar": "nanti", "skr": "sekarang", "skrg": "sekarang", "skg": "sekarang",
      "kmrn": "kemarin", "kemrn": "kemarin", "kmarin": "kemarin",
      "aja": "saja", "aj": "saja", "sj": "saja",
      "nih": "ini", "nie": "ini", "ni": "ini", "tuh": "itu", "gtu": "begitu", "gitu": "begitu",
      "trs": "terus", "trus": "terus",
      "yg": "yang", "utk": "untuk", "dlm": "dalam", "dr": "dari", "dg": "dengan", "jd": "jadi", "jg": "juga",
      "krn": "karena", "tp": "tetapi", "tpi": "tetapi", "sm": "sama", "thd": "terhadap",
      "dll": "dan lain-lain", "dsb": "dan sebagainya", "dst": "dan seterusnya",
      "banget": "sekali", "bgt": "sekali", "sgt": "sangat", "sngt": "sangat",
      "lg": "sedang", "sdg": "sedang",
      "dl": "dulu", "pls": "tolong", "tolongin": "tolong", "plis": "tolong",
      "wkwk": "tertawa", "wkwkwk": "tertawa", "hehe": "tertawa kecil", "hihi": "tertawa kecil",
      "btw": "ngomong-ngomong", "imo": "menurut saya", "imho": "menurut saya", "cmiiw": "koreksi jika saya salah",
      "idk": "saya tidak tahu", "jk": "hanya bercanda",
      "ok": "baik", "oke": "baik", "okey": "baik", "sip": "baik",
      "ciyus": "serius", "serem": "menyeramkan",
      "kl": "kalau", "klo": "kalau", "klu": "kalau",
      "spy": "supaya", "spya": "supaya",
      "bbrp": "beberapa", "tsb": "tersebut", "trsbt": "tersebut",
      "dpt": "dapat", "bs": "bisa", "bsa": "bisa",
      "stlh": "setelah", "sblm": "sebelum",

      "mnj": "manajemen", "man": "manajemen", "mgt": "management",
      "org": "organisasi", "org2": "organisasi", "orgzt": "organisasi",
      "str": "struktur", "stkt": "struktur",
      "ldr": "leader", "ldrshp": "leadership", "pimp": "pimpinan", "pemimp": "pemimpin",
      "pln": "perencanaan", "renc": "perencanaan", "plan": "perencanaan",
      "orgz": "organizing", "orgzn": "organisasi",
      "dir": "directing", "pgn": "pengarahan",
      "cnt": "control", "cont": "control", "ctrl": "kontrol", "pengend": "pengendalian",
      "eff": "efisiensi", "effct": "efektivitas",
      "sdm": "sumber daya manusia", "hr": "human resource", "hrd": "human resource development",
      "res": "resource", "rsc": "resource", "sda": "sumber daya alam",
      "inv": "investasi", "invt": "investasi",
      "pmas": "pemasaran", "mkt": "marketing", "mktg": "marketing",
      "prod": "produksi", "prdks": "produksi",
      "fin": "finance", "keu": "keuangan", "akut": "akuntansi", "acct": "akuntansi",
      "ris": "risiko", "rsko": "risiko",
      "anal": "analisis", "eval": "evaluasi",
      "strtg": "strategi", "stg": "strategi",
      "ops": "operasi", "opr": "operasional", "oprs": "operasional",
      "bsc": "balanced scorecard", "swot": "analisis swot", "pest": "analisis pest",
      "csr": "corporate social responsibility", "gcn": "good corporate governance",
      "qm": "quality management", "iso": "standar iso",
      "kpi": "key performance indicator", "indik": "indikator",
      "knowl": "knowledge management", "kmgt": "knowledge management",
      "chg": "change management", "innv": "inovasi",
      "cnfl": "konflik", "cnflt": "konflik",
      "krj": "kerja", "tm": "tim", "tmwrk": "kerja sama tim",
      "cst": "cost", "faktorfaktor": "faktor", "bya": "biaya",
      "val": "nilai", "valu": "value",
      "proj": "proyek", "prjk": "proyek",

      "med": "medis", "obat2": "obat-obatan", "rs": "rumah sakit",
      "dok": "dokter", "drg": "dokter gigi", "prof": "profesor",
      "pt": "perguruan tinggi", "univ": "universitas", "fak": "fakultas",
      "skripsi": "skripsi", "tesis": "tesis", "disertasi": "disertasi",
      "mhs": "mahasiswa", "mhsw": "mahasiswa"
  }


  pattern = r'\b(' + '|'.join(re.escape(key) for key in contractions_dict.keys()) + r')\b'

  def replace_match(match):
      return contractions_dict[match.group(0).lower()]

  expanded_text = re.sub(pattern, replace_match, text, flags=re.IGNORECASE)
  return expanded_text


tempo_df["isi_berita_expanded"] = tempo_df["isi_berita_stemmed"].apply(
    lambda tokens: expand_indonesian_contractions(" ".join(tokens)).split()
)


In [16]:
print("\nBerita - Isi berita (expanded):")
tempo_df[["isi_berita_stemmed", "isi_berita_expanded"]].head(10)


Berita - Isi berita (expanded):


,isi_berita_stemmed,isi_berita_expanded
0,"[menteri, koordinator, menko, bidang, ekonomi,...","[menteri, koordinator, menko, bidang, ekonomi,..."
1,"[rangka, riah, pt, surabaya, industrial, estat...","[rangka, riah, perguruan, tinggi, surabaya, in..."
2,"[wacana, presiden, prabowo, subianto, bentuk, ...","[wacana, presiden, prabowo, subianto, bentuk, ..."
3,"[bpjs, ketenagakerjaan, menteri, agama, kemena...","[bpjs, ketenagakerjaan, menteri, agama, kemena..."
4,"[perintah, bentuk, satu, tugas, putus, hubung,...","[perintah, bentuk, satu, tugas, putus, hubung,..."
5,"[menko, bidang, infrastruktur, bangun, wilayah...","[menko, bidang, infrastruktur, bangun, wilayah..."
6,"[kepala, badan, gizi, nasional, dad, hindayana...","[kepala, badan, gizi, nasional, dad, hindayana..."
7,"[menteri, koordinator, bidang, pangan, zulkifl...","[menteri, koordinator, bidang, pangan, zulkifl..."
8,"[uji, coba, alias, commissioning, bangkit, lis...","[uji, coba, alias, commissioning, bangkit, lis..."
9,"[anak, crazy, rich, usaha, sawit, kalimantan, ...","[anak, crazy, rich, usaha, sawit, kalimantan, ..."


In [ ]:
# Inisialisasi SpellChecker kosong
spell = SpellChecker(language=None)

# Load kamus Indonesia dari file
with open("00-indonesian-wordlist.lst", "r", encoding="latin-1") as f:
    indo_words = [line.strip() for line in f.readlines()]

spell.word_frequency.load_words(indo_words)

# Fungsi untuk koreksi kata
def correct_word(word):
    corr = spell.correction(word)
    return corr if corr is not None else word

# Terapkan spellcheck ke setiap baris dengan progress bar
corrected_texts = []
for tokens in tqdm(tempo_df["isi_berita_expanded"], desc="Spellchecking", unit="row"):
    corrected = [correct_word(word) for word in tokens]
    corrected_texts.append(corrected)

tempo_df["isi_berita_spellchecked"] = corrected_texts

Spellchecking:  90%|█████████ | 1354/1500 [6:18:44<37:14, 15.31s/row]

In [ ]:
print("\nBerita (Cek Ejaan):")
tempo_df[["isi_berita_expanded", "isi_berita_spellchecked"]].head(10)

## Pilih kolom yang ingin disimpan

In [ ]:
cols_to_save = [
    "no",
    "judul",
    "berita",
    "tanggal",
    "kategori",
    "link",
    "isi_berita_clean",
    "isi_berita_tokens",
    "isi_berita_filtered",
    "isi_berita_stemmed",
    "isi_berita_expanded",
    "isi_berita_spellchecked"
]

# Simpan ke CSV
tempo_df[cols_to_save].to_csv("tempo_processed.csv", index=False, encoding="utf-8-sig")

print("File berhasil disimpan sebagai tempo_processed.csv")


In [ ]:
tempo_df_prs= pd.read_csv("tempo_processed.csv")
tempo_df_prs

In [ ]:
corpus = []
for col in tempo_df['isi_berita_clean']:
    word_list = col.split(" ")
    corpus.append(word_list)

# Word2Vec dengan ukuran vektor 50
model = Word2Vec(corpus, min_count=1, vector_size=50, window=5, sg=0)


In [ ]:
print("Kata mirip dengan 'presiden':")
print(model.wv.most_similar('presiden', topn=5))

print("\nKata mirip dengan 'data':")
print(model.wv.most_similar('data', topn=5))

# contoh cosmul
print("\nCosmul (presiden + sistem - data):")
print(model.wv.most_similar_cosmul(positive=['presiden', 'sistem'], negative=['data'], topn=5))

# doesnt_match: cari kata yang tidak sesuai konteks
print("\nKata yang tidak cocok dalam ['presiden', 'data', 'sistem', 'informasi']:")
print(model.wv.doesnt_match("presiden data sistem informasi".split()))

# save embeddings
filename = 'pta_embeddings.txt'
model.wv.save_word2vec_format(filename, binary=False)
print(f"\nEmbeddings disimpan ke {filename}")

## Klasifikasi LDA

In [ ]:
if 'isi_berita_tokens' not in tempo_df.columns:
    tempo_df['isi_berita_tokens'] = tempo_df['isi_berita_clean'].apply(lambda x: str(x).split())

# Dataset teks tokenized
texts = tempo_df['isi_berita_tokens']

# Buat dictionary dan corpus
dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

# === Evaluasi jumlah topik optimal ===
coherence_values = []
model_list = []
topic_range = range(2, 130, 10)  # misalnya 2–120 topik

for num_topics in topic_range:
    model = LdaModel(
        corpus=corpus,
        num_topics=num_topics,
        id2word=dictionary,
        random_state=42,
        passes=10,
        iterations=100
    )
    model_list.append(model)
    coherencemodel = CoherenceModel(model=model, texts=texts, dictionary=dictionary, coherence='c_v')
    coherence_values.append(coherencemodel.get_coherence())

# === Visualisasi Coherence Score ===
plt.figure(figsize=(8, 5))
plt.plot(topic_range, coherence_values, marker='o', linestyle='--')
plt.xlabel("Jumlah Topik", fontsize=12)
plt.ylabel("Coherence Score", fontsize=12)
plt.title("Perbandingan Coherence Score terhadap Jumlah Topik", fontsize=14)
plt.grid(True)
plt.show()

# === Model terbaik ===
optimal_model = model_list[coherence_values.index(max(coherence_values))]
optimal_topics = topic_range[coherence_values.index(max(coherence_values))]

print("✅ Model terbaik memiliki", optimal_topics, "topik")
print("Coherence Score terbaik:", max(coherence_values))



In [ ]:
for idx, topic in optimal_model.print_topics(-1):
    print(f"\nTopik {idx+1}: {topic}")

In [ ]:
dominant_topics = []

for i, doc_bow in enumerate(corpus):
    topics_in_doc = optimal_model.get_document_topics(doc_bow, minimum_probability=0)
    topics_in_doc = sorted(topics_in_doc, key=lambda x: x[1], reverse=True)
    topic_num, prop = topics_in_doc[0]
    dominant_topics.append((topic_num + 1, round(prop, 3)))

df_dominant = pd.DataFrame(dominant_topics, columns=["Topik Dominan", "Proporsi"])

df_result = pd.concat([tempo_df.reset_index(drop=True), df_dominant.reset_index(drop=True)], axis=1)

display(df_result[["judul", "kategori", "Topik Dominan", "Proporsi", "isi_berita_clean"]])

In [ ]:
topic_distribution = [optimal_model.get_document_topics(doc, minimum_probability=0) for doc in corpus]
n_docs = min(10, len(topic_distribution))
data = np.array([[prob for _, prob in topic_distribution[i]] for i in range(n_docs)])

# Plot stacked bar
fig, ax = plt.subplots(figsize=(12, 6))
bottom = np.zeros(n_docs)

for k in range(data.shape[1]):
    ax.bar(range(n_docs), data[:, k], bottom=bottom)
    bottom += data[:, k]

# Judul dan label sumbu
ax.set_title("Distribusi Topik per Dokumen", fontsize=14)
ax.set_xlabel("Indeks Dokumen", fontsize=12)
ax.set_ylabel("Proporsi Topik", fontsize=12)

# Tambahkan legenda di luar area plot agar rapi
ax.legend(bbox_to_anchor=(1.05, 1),)
plt.tight_layout()
plt.show()


In [ ]:
topic_distribution = [optimal_model.get_document_topics(doc, minimum_probability=0) for doc in corpus]

num_docs_to_show = 10
topic_matrix = np.array([[score for _, score in doc] for doc in topic_distribution[:num_docs_to_show]])

plt.figure(figsize=(10, 6))
plt.imshow(topic_matrix, cmap='coolwarm', aspect='auto')
plt.colorbar(label='Proporsi Topik')
plt.xlabel('Topik')
plt.ylabel('Dokumen')
plt.title('Distribusi Topik pada Beberapa Dokumen Pertama')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer # Keep import in case needed later or for comparison

# Assuming optimal_model from LDA is available and corpus is defined

# Get topic distributions for each document from the optimal LDA model
# minimum_probability=0 ensures we get a score for all topics for each document
topic_distributions = [optimal_model.get_document_topics(doc, minimum_probability=0) for doc in corpus]

# Convert the list of topic distributions (list of tuples) into a numpy array
# Each row is a document, each column is a topic
# The order of topics in the tuple list matches the topic index
X_lda = np.array([[score for topic, score in sorted(doc_topics)] for doc_topics in topic_distributions])

# Get the labels (kategori)
y = df_result["kategori"].astype(str)   # Use 'kategori' column as labels

# Split the data using the LDA topic distributions as features
X_train, X_test, y_train, y_test = train_test_split(
    X_lda, y, test_size=0.2, random_state=42, stratify=y
)

print("Data berhasil dibagi menggunakan fitur dari distribusi topik LDA.")
print("Bentuk X_train:", X_train.shape)
print("Bentuk X_test:", X_test.shape)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)

print("Model Naive Bayes dilatih menggunakan fitur distribusi topik LDA.")

In [ ]:
def evaluasi_model(y_true, y_pred, model_name):
    print(f"\n=== Evaluasi Model: {model_name} ===")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average='weighted'))
    print("Recall   :", recall_score(y_true, y_pred, average='weighted'))
    print("\nClassification Report:\n", classification_report(y_true, y_pred))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
print("### 🔍 Evaluasi Model Naive Bayes ###")
evaluasi_model(y_test, y_pred_nb, "Naive Bayes")

In [ ]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import seaborn as sns

svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

# softmax_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
# softmax_model.fit(X_train, y_train)
# y_pred_softmax = softmax_model.predict(X_test)

print("Model SVM dilatih menggunakan fitur distribusi topik LDA.")
# print("Model Softmax Regression dilatih menggunakan fitur distribusi topik LDA.") # Uncomment if using Softmax

In [ ]:
def evaluasi_model(y_true, y_pred, model_name):
    print(f"\n=== Evaluasi Model: {model_name} ===")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average='weighted'))
    print("Recall   :", recall_score(y_true, y_pred, average='weighted'))
    print("\nClassification Report:\n", classification_report(y_true, y_pred))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
print("### 🔍 Evaluasi Model SVM ###")
evaluasi_model(y_test, y_pred_svm, "SVM")